# LLM v3 Model Evaluator

Inspect one saved `llm_v3` run on the validation or test split and compare it against fixed-chain baselines plus per-circuit oracles.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "machine_learning").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from machine_learning.LLM.llm_v3.analysis_utils import (
    build_model_evaluator_tables,
    draw_chain_usage_heatmap,
    draw_confidence_regret_scatter,
    draw_family_original_rate,
    draw_metric_panel,
    horizontal_bar,
    load_summary,
    resolve_latest_run,
    save_figure,
)

LLM_DIR = PROJECT_ROOT / "machine_learning" / "LLM" / "llm_v3"
ARTIFACT_ROOT = LLM_DIR / "artifacts"
FIGURE_ROOT = LLM_DIR / "model_evaluator_figures"


In [ ]:
EVAL_SPLIT = "test"  # "validation" or "test"
RUN_DIR = "./machine_learning/LLM/llm_v3/artifacts/full_system_test_2q_only"  # Set to a specific saved run directory, or leave as None for newest split-matching run.
SAVE_FIGURES = True

run_dir = Path(RUN_DIR) if RUN_DIR else resolve_latest_run(ARTIFACT_ROOT, EVAL_SPLIT)
run_name = run_dir.name
figure_dir = FIGURE_ROOT / run_name

print(f"Evaluating run: {run_dir}")
print(load_summary(run_dir))


In [ ]:
tables = build_model_evaluator_tables(run_dir)
manifest = tables["manifest"]
selected_rows = tables["selected_rows"]
selection_checks = tables["selection_checks"]
baseline_frames = tables["baseline_frames"]
comparison = tables["comparison"]
worst_regret_rows = tables["worst_regret_rows"]
family_summary = tables["family_summary"]

display(selection_checks)
display(comparison)
display(worst_regret_rows)
display(family_summary)
display(selected_rows.head())


In [ ]:
plot_frames = {str(selected_rows["method"].iloc[0]): selected_rows, **baseline_frames}
heatmap_fig = draw_chain_usage_heatmap(plot_frames, str(manifest["split"]))
if SAVE_FIGURES:
    save_figure(heatmap_fig, figure_dir, "selected_chain_counts_heatmap")
heatmap_fig.show()


In [ ]:
panel_fig = draw_metric_panel(
    comparison,
    title=f"LLM v3 vs Baselines on {str(manifest['split']).title()} Circuits",
)
if SAVE_FIGURES:
    save_figure(panel_fig, figure_dir, "baseline_comparison_panel")
panel_fig.show()

confidence_fig = draw_confidence_regret_scatter(selected_rows)
if SAVE_FIGURES:
    save_figure(confidence_fig, figure_dir, "confidence_vs_regret")
confidence_fig.show()

family_fig = draw_family_original_rate(family_summary)
if SAVE_FIGURES:
    save_figure(family_fig, figure_dir, "family_original_selection_rate")
family_fig.show()

objective_fig = horizontal_bar(
    comparison,
    x="avg_objective_score",
    xlabel="Average objective score",
    title=f"Average Objective Score on {str(manifest['split']).title()} Circuits",
)
if SAVE_FIGURES:
    save_figure(objective_fig, figure_dir, "avg_objective_score")
objective_fig.show()

regret_plot = comparison.dropna(subset=["avg_regret"]).copy()
if not regret_plot.empty:
    regret_fig = horizontal_bar(
        regret_plot,
        x="avg_regret",
        xlabel="Average regret",
        title=f"Average Regret on {str(manifest['split']).title()} Circuits",
        color="#C86D5B",
    )
    if SAVE_FIGURES:
        save_figure(regret_fig, figure_dir, "avg_regret")
    regret_fig.show()
